# nn.Module & Training Loop

Build your first neural network with nn.Module, understand how to organize model code, and write the canonical training loop that works with any PyTorch model.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/nn-module-training)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Your First nn.Module


In [ ]:
import torch
import torch.nn as nn

print("=" * 60)
print("DEFINING A SIMPLE NEURAL NETWORK")
print("=" * 60)

# Define a simple MLP
class SimpleNet(nn.Module):
    def __init__(self, input_size=28*28, hidden_size=128, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Flatten if needed
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create model
model = SimpleNet()
print(f"\nModel created: {model.__class__.__name__}")

# Print architecture
print(f"\nModel architecture:")
print(model)

print(f"\nParameter count:")
total_params = 0
for name, param in model.named_parameters():
    num_params = param.numel()
    total_params += num_params
    print(f"  {name:<20} {param.shape} — {num_params:,} parameters")
print(f"  Total: {total_params:,} parameters")

print("\n" + "=" * 60)
print("FORWARD PASS")
print("=" * 60)

# Create dummy input
x = torch.randn(32, 28*28)  # Batch of 32 MNIST images
print(f"\nInput shape: {x.shape}")

# Forward pass
y = model(x)
print(f"Output shape: {y.shape}")  # [32, 10] for 10 classes

print("\n" + "=" * 60)
print("DEVICE PLACEMENT")
print("=" * 60)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Move to device
model = model.to(device)
x = x.to(device)
y = model(x)
print(f"✓ Model and data on {device}")
print(f"Output device: {y.device}")

print("\n" + "=" * 60)
print("ACTIVATION FUNCTIONS")
print("=" * 60)

# Test different activation functions
x = torch.randn(5, 10)
print(f"\nInput (first element): {x[0, 0].item():.4f}")

activations = {
    'ReLU': nn.ReLU(),
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
}

for name, fn in activations.items():
    y = fn(x)
    print(f"{name:10} output: {y[0, 0].item():.4f}")

print("\n" + "=" * 60)
print("STATE_DICT: Saving and Loading")
print("=" * 60)

# Save state_dict
state_dict = model.state_dict()
print(f"\nState dict keys: {list(state_dict.keys())}")

# Save to file (for demo, not actually saved)
# torch.save(state_dict, 'model.pth')

# Load into new model
model2 = SimpleNet()
model2.load_state_dict(state_dict)
print(f"✓ Loaded weights into new model")

# Verify weights are the same
weights_match = torch.allclose(
    model.fc1.weight,
    model2.fc1.weight
)
print(f"Weights match? {weights_match}")

### Lesson example: The Canonical Training Loop


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("CANONICAL TRAINING LOOP")
print("=" * 60)

# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\nUsing device: {device}")

# 1. Generate synthetic data
print("\n1. Generating synthetic data...")
torch.manual_seed(42)
X = torch.randn(1000, 20)
y = (X[:, 0] + X[:, 1] > 0).long()  # Binary classification

# Split: 800 train, 200 val
train_X, val_X = X[:800], X[800:]
train_y, val_y = y[:800], y[800:]

# Create dataloaders
train_dataset = TensorDataset(train_X, train_y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_X, val_y), batch_size=32, shuffle=False)
print(f"   Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# 2. Define model
print("\n2. Defining model...")
class SimpleNet(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, output_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleNet().to(device)
print(f"   Model: {sum(p.numel() for p in model.parameters())} parameters")

# 3. Setup training
print("\n3. Setting up optimizer and loss...")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# 4. Training loop
print("\n4. Training for 20 epochs...")
num_epochs = 20

for epoch in range(num_epochs):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()  # Clear old gradients
        logits = model(batch_x)  # Forward pass
        loss = criterion(logits, batch_y)  # Compute loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update parameters

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ========== VALIDATION ==========
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():  # No gradients during validation
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()

            preds = logits.argmax(dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += batch_y.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    scheduler.step()

    if (epoch + 1) % 5 == 0:
        print(f"   Epoch {epoch+1:2d}/{num_epochs} | "
              f"Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

print("\n✓ Training complete!")

---

## Exercise: Train a Classifier on MPS


Generate synthetic 2-class data, build a 2-layer MLP, train on MPS (or CPU if unavailable), and report final accuracy.

**Requirements:**
- Generate 400 samples in 2 balanced classes (use torch.randn and simple logic)
- Split into train (300) and validation (100) sets
- Build an MLP: input_size=20 → hidden=64 → output=2 (binary classification)
- Train for 20 epochs with batch_size=32, reporting loss every 5 epochs
- Use Adam optimizer with lr=1e-3
- Report final train and validation accuracy (should both be > 90%)
- Handle device selection gracefully (MPS or CPU)

**Hints:**
- Use `model.train()` in training loop, `model.eval()` in validation
- Use `torch.no_grad()` during validation (no need to compute gradients)
- Validation accuracy should be > 95% if training works correctly
- Use `torch.manual_seed(42)` for reproducibility
- Create a simple DataLoader with batch_size=32


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Generate synthetic data
torch.manual_seed(42)
num_samples = 400
X = torch.randn(num_samples, 20)
y = (X[:, 0] + X[:, 1] > 0).long()  # Binary classification

# Split into train/val
train_indices = torch.arange(300)
val_indices = torch.arange(300, 400)
X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[val_indices], y[val_indices]

# Create DataLoaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 2. Define MLP
class MLP(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, output_size=2):
        super().__init__()
        # TODO: Define two linear layers with ReLU activation between them

# 3. Training setup
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. Training loop
num_epochs = 20
for epoch in range(num_epochs):
    # TODO: Training step (forward, backward, step)
    # TODO: Validation step (no_grad, eval mode)
    # TODO: Print loss every 5 epochs

# TODO: Print final train and validation accuracy

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/nn-module-training) for the solution and discussion.
